# Embedding baseline: few-shot and zero-shot classification

Each augmented transcript (text with its tags) is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`, and the FGW distance is replaced by a distance between vectors. Ability and benevolence are evaluated separately.

* **Few-shot** ($m$ labeled examples per level): each level is represented by the mean embedding of its $m$ examples, and every other transcript gets the level of the nearest mean. The examples are drawn with stratified $k$-fold cross-validation: for each fold, the $m$ examples per level come from the other folds and the transcripts of the fold are classified. The folds and examples are saved in `splits/` so that the FGW and LLM methods use exactly the same ones.
* **Zero-shot** (unsupervised): $k$-means with $K = 3$ on the embeddings. Clusters carry no label, so each is mapped to a level with the Hungarian algorithm (the one-to-one assignment that maximizes agreement with the reference levels).

The reference level is either the **instructed** level (given to the LLM) or the **perceived** level (median of the participants' ratings).

In [ ]:
import os
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, adjusted_mutual_info_score, adjusted_rand_score,
                             confusion_matrix, f1_score, ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedKFold

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
SPLIT_DIR = "splits"            # folds and few-shot examples, shared with the other methods
OUTPUT_DIR = "results"

DIMENSIONS = ["ability", "benevolence"]
LEVELS = ["Low", "Medium", "High"]
REFERENCE = "instructed"        # "instructed" or "perceived"
# For REFERENCE = "perceived": csv with columns id, dimension, perceived (Low / Medium / High).
# Only the transcripts listed in it are evaluated.
PERCEIVED_FILE = None

M = 10                          # labeled examples per level (few-shot)
N_FOLDS = 5
SEED = 0
N_PER_LEVEL = None              # evaluate at most this many transcripts per level (None: all)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean"
N_KMEANS_INIT = 10              # k-means restarts

os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data

All `full_*` conditions with their instructed level. Each transcript gets an `id` = `<condition>#<row>` (the `generated_id` column is not unique across files).

In [ ]:
PREFIX = "generated_instructions_with_intention_explanation_"

frames = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, PREFIX + "full_*.csv"))):
    condition = os.path.basename(path)[len(PREFIX):-len(".csv")]
    if condition.endswith("ratedbyFGW"):
        continue
    data = pd.read_csv(path)
    for dimension in DIMENSIONS:
        if dimension not in data.columns:
            continue
        frames.append(pd.DataFrame({
            "id": [f"{condition}#{row}" for row in data.index],
            "condition": condition,
            "dimension": dimension,
            "text": data["instruction_with_intention"],
            "instructed": data[dimension].astype(str).str.split(" ").str[0],
        }))

df = pd.concat(frames, ignore_index=True).dropna(subset=["text"])
df = df[df["instructed"].isin(LEVELS)]

if REFERENCE == "perceived":
    perceived = pd.read_csv(PERCEIVED_FILE)
    df = df.merge(perceived[["id", "dimension", "perceived"]], on=["id", "dimension"])
df["level"] = df[REFERENCE]
df = df[df["level"].isin(LEVELS)]

if N_PER_LEVEL is not None:
    df = pd.concat([group.sample(min(len(group), N_PER_LEVEL), random_state=SEED)
                    for _, group in df.groupby(["dimension", "level"])])
df = df.reset_index(drop=True)
df.groupby(["dimension", "level"]).size().unstack()

## Embeddings

Cached in `results/embeddings.npz`, so each transcript is encoded once.

In [ ]:
cache_path = os.path.join(OUTPUT_DIR, "embeddings.npz")
cache = {}
if os.path.exists(cache_path):
    stored = np.load(cache_path, allow_pickle=True)
    cache = dict(zip(stored["ids"], stored["X"]))

texts = df.drop_duplicates("id").set_index("id")["text"]
missing = [i for i in texts.index if i not in cache]
if missing:
    model = SentenceTransformer(EMBEDDING_MODEL)
    vectors = model.encode(texts[missing].tolist(), batch_size=64, show_progress_bar=True)
    cache.update(zip(missing, vectors))
    np.savez(cache_path, ids=np.array(list(cache), dtype=object), X=np.array(list(cache.values())))


def embed(ids):
    X = np.array([cache[i] for i in ids])
    if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
        X = X / np.linalg.norm(X, axis=1, keepdims=True)
    return X

## Folds and few-shot examples

Created once per dimension and reference, then reloaded: every method must use the same split.

In [ ]:
def load_or_create_split(data, dimension):
    # Returns folds (id, level, fold) and examples (fold, level, id).
    folds_path = os.path.join(SPLIT_DIR, f"folds_{dimension}_{REFERENCE}.csv")
    examples_path = os.path.join(SPLIT_DIR, f"examples_{dimension}_{REFERENCE}_m{M}.csv")
    if os.path.exists(folds_path) and os.path.exists(examples_path):
        folds, examples = pd.read_csv(folds_path), pd.read_csv(examples_path)
        assert set(folds["id"]) == set(data["id"]), f"{folds_path} was made on other data"
        return folds, examples

    folds = data[["id", "level"]].copy()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (_, test) in enumerate(skf.split(folds, folds["level"])):
        folds.loc[folds.index[test], "fold"] = fold
    folds["fold"] = folds["fold"].astype(int)

    rng = np.random.default_rng(SEED)
    rows = []
    for fold in range(N_FOLDS):
        train = folds[folds["fold"] != fold]
        for level in LEVELS:
            chosen = rng.choice(train.loc[train["level"] == level, "id"].to_numpy(), M, replace=False)
            rows += [{"fold": fold, "level": level, "id": i} for i in chosen]
    examples = pd.DataFrame(rows)

    folds.to_csv(folds_path, index=False)
    examples.to_csv(examples_path, index=False)
    return folds, examples

## Few-shot: nearest mean embedding

In [ ]:
def few_shot(data, folds, examples):
    # Predicted level of every transcript, from the examples of its own fold.
    predictions = []
    for fold in range(N_FOLDS):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        centroids = np.array([embed(support.loc[support["level"] == level, "id"]).mean(axis=0)
                              for level in LEVELS])
        X = embed(test["id"])
        distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        predictions.append(pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                                         "pred": np.array(LEVELS)[distances.argmin(axis=1)]}))
    return pd.concat(predictions, ignore_index=True)

## Zero-shot: $k$-means and Hungarian mapping

In [ ]:
def zero_shot(data):
    X = embed(data["id"])
    clusters = KMeans(n_clusters=len(LEVELS), n_init=N_KMEANS_INIT, random_state=SEED).fit_predict(X)
    true = data["level"].to_numpy()

    # contingency[c, l] = number of transcripts of cluster c with level l
    contingency = np.array([[np.sum((clusters == c) & (true == level)) for level in LEVELS]
                            for c in range(len(LEVELS))])
    cluster_idx, level_idx = linear_sum_assignment(-contingency)
    mapping = {c: LEVELS[l] for c, l in zip(cluster_idx, level_idx)}
    return pd.DataFrame({"id": data["id"], "true": true, "cluster": clusters,
                         "pred": [mapping[c] for c in clusters]})

## Run

In [ ]:
def scores(pred):
    return {"accuracy": accuracy_score(pred["true"], pred["pred"]),
            "macro_f1": f1_score(pred["true"], pred["pred"], labels=LEVELS, average="macro")}


records, all_predictions = [], {}
for dimension in DIMENSIONS:
    data = df[df["dimension"] == dimension]
    folds, examples = load_or_create_split(data, dimension)

    pred = few_shot(data, folds, examples)
    all_predictions[(dimension, "few-shot")] = pred
    for fold, pred_fold in pred.groupby("fold"):
        records.append({"dimension": dimension, "setting": "few-shot", "fold": fold, **scores(pred_fold)})
    records.append({"dimension": dimension, "setting": "few-shot", "fold": "all", **scores(pred)})

    pred = zero_shot(data)
    all_predictions[(dimension, "zero-shot")] = pred
    records.append({"dimension": dimension, "setting": "zero-shot", "fold": "all", **scores(pred),
                    "ARI": adjusted_rand_score(pred["true"], pred["cluster"]),
                    "AMI": adjusted_mutual_info_score(pred["true"], pred["cluster"])})

results = pd.DataFrame(records)
results.insert(0, "method", "embeddings")
results.insert(1, "reference", REFERENCE)
results.to_csv(os.path.join(OUTPUT_DIR, f"embedding_baseline_{REFERENCE}.csv"), index=False)
for (dimension, setting), pred in all_predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"embedding_predictions_{dimension}_{setting}_{REFERENCE}.csv"),
                index=False)

## Results

Few-shot: mean ± standard deviation over folds, and on all folds pooled. Chance level is 1/3.

In [ ]:
per_fold = results[results["fold"] != "all"]
summary = (per_fold.groupby(["dimension", "setting"])[["accuracy", "macro_f1"]]
           .agg(["mean", "std"]).round(3))
summary

In [ ]:
results[results["fold"] == "all"].drop(columns="fold").round(3)

In [ ]:
fig, axes = plt.subplots(len(DIMENSIONS), 2, figsize=(10, 4.5 * len(DIMENSIONS)), squeeze=False)
for row, dimension in enumerate(DIMENSIONS):
    for col, setting in enumerate(["few-shot", "zero-shot"]):
        pred = all_predictions[(dimension, setting)]
        cm = confusion_matrix(pred["true"], pred["pred"], labels=LEVELS, normalize="true")
        ConfusionMatrixDisplay(cm, display_labels=LEVELS).plot(ax=axes[row, col], cmap="Blues",
                                                               values_format=".2f", colorbar=False)
        axes[row, col].set_title(f"{dimension}, {setting} (accuracy = {scores(pred)['accuracy']:.2f})")
        axes[row, col].set_xlabel("predicted level")
        axes[row, col].set_ylabel(f"{REFERENCE} level")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, f"embedding_confusion_{REFERENCE}.png"), dpi=300)
plt.show()